# CS 198/199 
# Joaquin B. Salvador
# Explainable AI for Post-COVID Psychological Profile Prediction

## 2.2
### Torch

# Data Preparation

We currently have a dataset from an online survey conducted on the Japanese population when the number of COVID-19-positive cases was high and when the COVID-19 pandemic was officially declared to be over. The dataset has 92 attributes, including socio-demographic, COVID-19-related, mental health, and coping behavior information from 2,659 respondents.

In [1]:
# -------------------------------------
# Library configuration
# -------------------------------------
# Standard libraries
import numpy as np
import pandas as pd

# ML libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torchmetrics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# XAI libraries
import shap

In [2]:
# -------------------------------------
# Settings
# -------------------------------------
random_state = 42

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using cpu device


## Data preprocessing

In [4]:
# Loading dataset
df = pd.read_csv('../data/data.csv')

In [5]:
# Displaying dataset
df

,SAMPLEID,ANSWERDATE,5_SEX,5_AGE,5_PREFECTURE,5_MARRIED,5_CHILD,5_HINCOME,5_PINCOME,5_JOB,...,5_AUDIT_group,6_K6_total,6_PHQ9_total,6_GAD7_total,6_SSS8_total,6_PTGI-X(Q7_22_27),6_SHS_total,6_UCLA_total,6_LSNS6_total,6_AUDIT
0,1106,2022/05/13-19:39:52,1.0,67.0,14.0,2.0,2.0,3.0,1.0,12.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1759,2022/05/14-08:44:24,1.0,57.0,40.0,2.0,2.0,5.0,3.0,3.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1916,2022/05/13-20:19:11,1.0,62.0,11.0,2.0,2.0,2.0,2.0,3.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3474,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3793,2022/05/14-18:23:23,1.0,55.0,40.0,2.0,2.0,4.0,3.0,4.0,...,0.0,12.0,15.0,14.0,18.0,19.0,4.0,24.0,4.0,6.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24045,30063086,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24046,30081779,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24047,30093317,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24048,30094439,2022/05/13-21:35:19,2.0,34.0,28.0,2.0,1.0,10.0,1.0,8.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# Sorting dataset based on SAMPLEID
df_sorted = df.sort_values(by=['SAMPLEID'])

In [7]:
# Displaying sorted dataset
df_sorted

,SAMPLEID,ANSWERDATE,5_SEX,5_AGE,5_PREFECTURE,5_MARRIED,5_CHILD,5_HINCOME,5_PINCOME,5_JOB,...,5_AUDIT_group,6_K6_total,6_PHQ9_total,6_GAD7_total,6_SSS8_total,6_PTGI-X(Q7_22_27),6_SHS_total,6_UCLA_total,6_LSNS6_total,6_AUDIT
0,1106,2022/05/13-19:39:52,1.0,67.0,14.0,2.0,2.0,3.0,1.0,12.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16161,1171,2022/05/13-19:35:52,1.0,59.0,14.0,2.0,2.0,6.0,5.0,3.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16162,1263,2022/05/13-19:44:02,1.0,73.0,14.0,2.0,2.0,2.0,2.0,12.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14191,1346,2022/05/13-19:00:17,1.0,52.0,26.0,2.0,2.0,3.0,3.0,2.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18612,1398,2022/05/13-20:26:45,1.0,52.0,27.0,1.0,1.0,3.0,3.0,6.0,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17614,30094793,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16160,30094932,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24049,30095042,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20047,30095074,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# Dropping empty rows from sorted dataset
df_cleaned = df_sorted.dropna()

In [9]:
# Displaying cleaned dataset
df_cleaned

,SAMPLEID,ANSWERDATE,5_SEX,5_AGE,5_PREFECTURE,5_MARRIED,5_CHILD,5_HINCOME,5_PINCOME,5_JOB,...,5_AUDIT_group,6_K6_total,6_PHQ9_total,6_GAD7_total,6_SSS8_total,6_PTGI-X(Q7_22_27),6_SHS_total,6_UCLA_total,6_LSNS6_total,6_AUDIT
4,3793,2022/05/14-18:23:23,1.0,55.0,40.0,2.0,2.0,4.0,3.0,4.0,...,0.0,12.0,15.0,14.0,18.0,19.0,4.00,24.0,4.0,6.0
8,9703,2022/05/14-07:51:04,2.0,68.0,28.0,2.0,2.0,8.0,1.0,8.0,...,0.0,1.0,0.0,0.0,1.0,3.0,4.50,11.0,18.0,0.0
11,11532,2022/05/14-12:53:15,1.0,50.0,13.0,1.0,1.0,3.0,3.0,3.0,...,0.0,1.0,2.0,0.0,1.0,0.0,4.00,32.0,0.0,0.0
15,15778,2022/05/13-19:09:38,1.0,64.0,27.0,2.0,2.0,5.0,3.0,3.0,...,0.0,0.0,0.0,0.0,0.0,18.0,5.50,21.0,13.0,0.0
19,18713,2022/05/13-22:01:06,1.0,61.0,13.0,2.0,2.0,10.0,10.0,2.0,...,0.0,0.0,2.0,0.0,0.0,15.0,5.25,22.0,18.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11204,26475923,2022/05/15-01:24:36,1.0,55.0,27.0,1.0,2.0,2.0,2.0,3.0,...,0.0,12.0,4.0,7.0,13.0,18.0,4.00,25.0,6.0,6.0
11208,26476814,2022/05/13-19:35:37,1.0,53.0,13.0,2.0,2.0,7.0,7.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,4.00,25.0,0.0,0.0
11209,26476898,2022/05/13-21:36:28,2.0,48.0,40.0,2.0,1.0,5.0,1.0,8.0,...,0.0,13.0,6.0,0.0,5.0,12.0,3.75,26.0,4.0,0.0
11288,26520090,2022/05/13-19:39:54,2.0,31.0,13.0,1.0,1.0,3.0,3.0,3.0,...,0.0,2.0,3.0,2.0,14.0,12.0,4.00,27.0,3.0,0.0


In [10]:
# Gathering column names
column_names = df_cleaned.columns.tolist()

In [11]:
# Displaying column names
column_names

['SAMPLEID',
 'ANSWERDATE',
 '5_SEX',
 '5_AGE',
 '5_PREFECTURE',
 '5_MARRIED',
 '5_CHILD',
 '5_HINCOME',
 '5_PINCOME',
 '5_JOB',
 '5_STUDENT',
 '5_Number_fam',
 '5_SchoolGrade',
 '5_YearsEnrolled',
 '5_VacNum',
 '5_UkraineMoviePicture',
 '5_UkraineReadListen',
 '5_Med_self',
 '5_Medself_covid',
 '5_Med_fam',
 '5_Medfam_covid',
 '5_Current_physical',
 '5_Past_physical',
 '5_Current_mental',
 '5_Past_mental',
 '5_Current_covid19',
 '5_Past_covid19',
 '5_K6_total',
 '5_PHQ9_total',
 '5_GAD7_total',
 '5_SSS8_total',
 '5_PTGI-X(Q7_39_44)',
 '5_MAIA_1',
 '5_MAIA_2',
 '5_MAIA_3',
 '5_MAIA_4',
 '5_MAIA_5',
 '5_MAIA_6',
 '5_MAIA_7',
 '5_MAIA_8',
 '5_SHS_total',
 '5_UCLA_total',
 '5_LSNS6_total',
 '5_AUDIT',
 '5_Exercise',
 '5_HealthyDiet',
 '5_FavoriteActivity',
 '5_Interaction_offline',
 '5_Interaction_online',
 '5_PB_Continuous',
 '5_PB_Altruistic',
 '5_PB_Avoidant',
 '5_Trust_gov',
 '5_Trust_SM',
 '5_Vaccination_will',
 '5_PB_understanding',
 '5_Optimism',
 '5_HealthySleep',
 '5_Deterioratio

In [14]:
# Setting targets
df_targets = df_cleaned[['6_K6_total',
 '6_PHQ9_total',
 '6_GAD7_total',
 '6_SSS8_total',
 '6_PTGI-X(Q7_22_27)',
 '6_SHS_total',
 '6_UCLA_total',
 '6_LSNS6_total',
 '6_AUDIT']]

num_targets = len(df_targets.columns.tolist())

In [15]:
# Setting features
df_features = df_cleaned.drop([
 'SAMPLEID',
 'ANSWERDATE',
 '5_PREFECTURE',
 '6_K6_total',
 '6_PHQ9_total',
 '6_GAD7_total',
 '6_SSS8_total',
 '6_PTGI-X(Q7_22_27)',
 '6_SHS_total',
 '6_UCLA_total',
 '6_LSNS6_total',
 '6_AUDIT'], axis=1)

num_features = len(df_features.columns.tolist())

# Building the model

Based on the following references:

https://www.kaggle.com/code/schmiddey/multiclass-classification-with-pytorch
https://www.youtube.com/watch?v=iWdVXAwurXs

In [16]:
# Setting torch seed for consistency
torch.manual_seed(random_state)

In [17]:
X = df_features.to_numpy()
y = df_targets.to_numpy()

In [18]:
# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=random_state)

In [19]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [20]:
# Assuming X_train, y_train, X_test, y_test are NumPy arrays
X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
print(X_train_tensor.shape)

y_train_tensor = torch.tensor(np.argmax(y_train, axis=1), dtype=torch.long).to(device)
print(y_train_tensor.shape)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
print(X_test_tensor.shape)

y_test_tensor = torch.tensor(np.argmax(y_test, axis=1), dtype=torch.long).to(device)
print(y_test_tensor.shape)

torch.Size([1861, 87])
torch.Size([1861])
torch.Size([798, 87])
torch.Size([798])


In [21]:
# Follows specified initialization in the PyTorch tutorials
# https://pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html

# Leaky ReLU as per this post:
# https://stackoverflow.com/questions/69240517/what-is-the-best-choice-for-an-activation-function-in-case-of-small-sized-neural

class NeuralNetwork(nn.Module):
    def __init__(self, num_features, num_targets, dropout_rate):
        super(NeuralNetwork, self).__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            # -----------------------------------------
            # Input layer
            nn.Linear(num_features, num_features),
            nn.BatchNorm1d(num_features),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            # -----------------------------------------

            # -----------------------------------------
            # Hidden Layers        
            nn.Linear(num_features, num_features // 2),
            nn.BatchNorm1d(num_features // 2),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),

            nn.Linear(num_features // 2, num_features // 4),
            nn.BatchNorm1d(num_features // 4),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            
            nn.Linear(num_features // 4, num_features // 8),
            nn.BatchNorm1d(num_features // 8),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),

            nn.Linear(num_features // 8, num_features // 16),
            nn.BatchNorm1d(num_features // 16),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            
            nn.Linear(num_features // 16, num_features // 32),
            nn.BatchNorm1d(num_features // 32),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            
            # -----------------------------------------

            # -----------------------------------------
            # Output Layer
            nn.Linear(num_features // 32, num_targets)
            # -----------------------------------------
        )

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            module.weight.data.normal_(mean=0.0, std=1)
            if module.bias is not None:
                module.bias.data.zero_()

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [22]:
# Usage of softmax as per this post
# https://www.turing.com/kb/softmax-multiclass-neural-networks

learning_rate = 0.01
num_epochs = 1000
dropout_rate = 0.6
     
model = NeuralNetwork(num_features, num_targets, dropout_rate).to(device)
logits = model(X_train_tensor)
probabilities = torch.softmax(logits, dim=1)
model.apply(model._init_weights)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=87, out_features=87, bias=True)
    (1): BatchNorm1d(87, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.6, inplace=False)
    (4): Linear(in_features=87, out_features=43, bias=True)
    (5): BatchNorm1d(43, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.6, inplace=False)
    (8): Linear(in_features=43, out_features=21, bias=True)
    (9): BatchNorm1d(21, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.6, inplace=False)
    (12): Linear(in_features=21, out_features=10, bias=True)
    (13): BatchNorm1d(10, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (14): ReLU()
    (15): Dropout(p=0.6, inplace=False)
    (16): Linear(in_features=10, out_features=5, bias=True)
    (17): BatchNorm1d(

In [23]:
# Loss function
loss_function = nn.CrossEntropyLoss()

#optimizer = optim.Adam(model.parameters(), lr=learning_rate)
optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)

# Instantiate accuracy metric
accuracy = torchmetrics.Accuracy("multiclass", num_classes=num_features).to(device)

In [24]:
# Referred to this Kaggle post
# https://www.kaggle.com/code/dietzschenostoevsky/multiclass-classification-sgdvsadam-optimizer

for epoch in range(num_epochs):
    model.train()
    
    # 1. Forward pass
    y_logits = model(X_train_tensor)
    y_pred = torch.softmax(y_logits, dim=1).argmax(dim=1)
    
    # 2. Calculating loss and accuracy
    loss = loss_function(y_logits, y_train_tensor)
    acc = accuracy(preds=y_pred, target=y_train_tensor)
    
    # 3. Optimizer zero grad
    optimizer.zero_grad()
    
    # 4. Loss backward
    loss.backward()
    
    # 5. Optimizer step
    optimizer.step()
    
    # 6. Testing
    model.eval()
    with torch.inference_mode():
        # 1. Forward Pass
        test_logits = model(X_test_tensor)
        test_pred = torch.softmax(test_logits, dim=1).argmax(dim=1)
        
        # 2. Test loss and accuracy
        test_loss = loss_function(test_logits, y_test_tensor)
        test_acc = accuracy(preds=test_pred, target=y_test_tensor)

    # Printouts
    if epoch % 100 == 0:
        # Compute training outputs and loss
        model.train()
        outputs = model(X_train_tensor)
        loss = loss_function(outputs, y_train_tensor)  # Ensure y_train_tensor has shape (N,)
        
        # Compute training accuracy
        _, predicted_labels = torch.max(outputs, 1)
        correct_predictions = (predicted_labels.cpu() == y_train_tensor.cpu()).sum().item()
        total_samples = len(y_train_tensor)
        train_accuracy = correct_predictions / total_samples  # Renamed to avoid conflict
        
        # Backpropagation and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    # Validation phase
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_test_tensor)
        val_loss = loss_function(val_outputs, y_test_tensor)
        
        _, val_predicted_labels = torch.max(val_outputs, 1)
        val_correct_predictions = (val_predicted_labels.cpu() == y_test_tensor.cpu()).sum().item()
        val_total_samples = len(y_test_tensor)
        val_accuracy = val_correct_predictions / val_total_samples
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch + 1}/{num_epochs}, '
              f'Train Loss: {loss.item():.4f}, Train Accuracy: {train_accuracy:.4f}, '
              f'Val Loss: {val_loss.item():.4f}, Val Accuracy: {val_accuracy:.4f}')


Epoch 10/1000, Train Loss: 2.1984, Train Accuracy: 0.4127, Val Loss: 1.9399, Val Accuracy: 0.7982
Epoch 20/1000, Train Loss: 1.6894, Train Accuracy: 0.4127, Val Loss: 1.5769, Val Accuracy: 0.7982
Epoch 30/1000, Train Loss: 1.3231, Train Accuracy: 0.4127, Val Loss: 1.2727, Val Accuracy: 0.7982
Epoch 40/1000, Train Loss: 1.1093, Train Accuracy: 0.4127, Val Loss: 1.0789, Val Accuracy: 0.7982
Epoch 50/1000, Train Loss: 0.9887, Train Accuracy: 0.4127, Val Loss: 0.9726, Val Accuracy: 0.7982
Epoch 60/1000, Train Loss: 0.9299, Train Accuracy: 0.4127, Val Loss: 0.9168, Val Accuracy: 0.7982
Epoch 70/1000, Train Loss: 0.8968, Train Accuracy: 0.4127, Val Loss: 0.8858, Val Accuracy: 0.7982
Epoch 80/1000, Train Loss: 0.8781, Train Accuracy: 0.4127, Val Loss: 0.8668, Val Accuracy: 0.7982
Epoch 90/1000, Train Loss: 0.8659, Train Accuracy: 0.4127, Val Loss: 0.8535, Val Accuracy: 0.7982
Epoch 100/1000, Train Loss: 0.8533, Train Accuracy: 0.4127, Val Loss: 0.8432, Val Accuracy: 0.7982
Epoch 110/1000, Tra

# Model Explanation

## Shapley Additive Explanations

In [ ]:
# Wrapper from this post
# https://github.com/shap/shap/issues/1067

def predict(input_values):
    input_values = torch.tensor(input_values, dtype=torch.float32)
    return np.array(model(input_values))


# Create the SHAP Kernel Explainer
with torch.no_grad():
    explainer = shap.KernelExplainer(model=predict, data=X_train)
    shap_values = explainer.shap_values(X_train)
    shap_test_values = explainer.shap_values(X_test)

Using 1861 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


  0%|          | 0/1861 [00:00<?, ?it/s]

  0%|          | 0/798 [00:00<?, ?it/s]

### Feature Importance

In [ ]:
with torch.no_grad():
    shap.summary_plot(shap_values, X_test, feature_names=column_names, plot_type='bar')

### Feature Effects

In [ ]:
with torch.no_grad():
    shap.summary_plot(shap_values, X_train, feature_names=column_names, cmap='coolwarm')

In [ ]:
with torch.no_grad():
    shap.summary_plot(shap_values, X_test, feature_names=column_names, cmap='coolwarm')

### Dependence

In [ ]:
print("Interaction values")

with torch.no_grad():
    shap_interaction_values = explainer.shap_interaction_values(X_test)
print(shap_interaction_values)

In [ ]:
sns.set_style("darkgrid")